In [1]:
import pandas as pd

# generate train data by running load_data.py
df = pd.read_parquet("./data/train.parquet")

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("stabilityai/stable-code-3b")

# sample to experiment with
TEXT = "def add(a: int, b)"

tokens = tokenizer.tokenize(TEXT)
encoded_tokens = tokenizer.encode(TEXT)


In [4]:
from typing import Literal

def get_best_device() -> Literal["mps", "cpu", "cuda"]:
    """
    Returns the best available torch device:
    Priority: CUDA > MPS > CPU
    """
    
    if torch.cuda.is_available():
        return "cuda"
    
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return "mps" # mac only
    
    return "cpu"

In [5]:
model = AutoModelForCausalLM.from_pretrained(
  "stabilityai/stable-code-3b",
  torch_dtype="auto",
)
model.to(get_best_device())



`torch_dtype` is deprecated! Use `dtype` instead!
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

StableLmForCausalLM(
  (model): StableLmModel(
    (embed_tokens): Embedding(50304, 2560)
    (layers): ModuleList(
      (0-31): 32 x StableLmDecoderLayer(
        (self_attn): StableLmSdpaAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=False)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=False)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=False)
          (o_proj): Linear(in_features=2560, out_features=2560, bias=False)
          (attention_dropout): Dropout(p=0.0, inplace=False)
          (rotary_emb): StableLmRotaryEmbedding()
        )
        (mlp): StableLmMLP(
          (gate_proj): Linear(in_features=2560, out_features=6912, bias=False)
          (up_proj): Linear(in_features=2560, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, element

In [6]:
inputs = tokenizer(TEXT, return_tensors="pt").to(model.device)
tokens = model.generate(
  **inputs,
  max_new_tokens=48,
  temperature=0.8,
  do_sample=True,
)
print(tokenizer.decode(tokens[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


def add(a: int, b) -> int:
    """
    add method for adding two numbers
    """
    return a + b


def test_add():
    """
    test add method to make sure that it adds two numbers
    
